NLSI 2D I/O Volterra system with separable 4D quadratic kernel

---

Kishore Kumar Tarafdar, Date: 06-06-2025


In [1]:
pwd

'/data1/kishoretarafdar/src.port/NLSI.v0'

In [5]:
!python --version

Python 3.12.7


        Disable GPU: Force tensorflow to select CPU

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="-1"    
import tensorflow as tf

2025-06-21 19:50:53.234035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750515653.255497 1488393 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750515653.262137 1488393 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-21 19:50:53.285086: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


        Select a GPU with a memory limit

In [1]:
import tensorflow as tf
print(f"TensorFlow version {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
len(gpus)

2025-06-21 19:54:21.813956: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750515861.835563 1489992 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750515861.842217 1489992 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-21 19:54:21.865439: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version 2.18.0
Num GPUs Available:  3


3

Select one GPU

        Restrict code to use a particular GPU...

In [2]:
# # include ../dirx 
mylibpath = [
    '/home/kishoretarafdar/bin',
    '/data1/kishoretarafdar/src.port/NSLI.v00/utils.Volterra'
    #'/home/k/PLAYGROUND10GB/SKULSTRIPpaper__'
    ]
import sys
[sys.path.insert(0,_) for _ in mylibpath]
del mylibpath

from tf_select_a_gpu import select_a_gpu

In [5]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [3]:
# select_gpu = gpus[gpu_id]
memory_limit = 32 #GB
select_a_gpu(gpus, gpu_id=2, memory_limit=memory_limit)
# del gpu_id, select_a_gpu, select_gpu

3 Physical GPUs available 
Selected 1 Logical GPU with 32 GB memory limit


I0000 00:00:1750515864.180801 1489992 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 32768 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:41:00.0, compute capability: 8.6


In [4]:
#%%
import tensorflow as tf
# from ConvNDv0 import ConvND

class NLSI2DVolterra(tf.keras.layers.Layer):
    """2D input-output NLSI system with 2nd order Volterra approximation 
    
    Limitation: Be careful with the # of filters

    --kkt@06-06-2025"""
    def __init__(
        self, 
        filters=1, 
        kernel_size=3,
        **kwargs):
        super(NLSI2DVolterra, self).__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
     

    def build(self, input_shape):
        self.inchannels = input_shape[-1]
        # self.filters = input_shape[-1]

        ## h0:= 0th order term
        # Initialize the bias (a0) for the numerator polynomial
        self.a0 = self.add_weight(shape=(self.filters,), initializer='zeros', trainable=True)#, name='h0_bias')

        ## h:= UIR kernle       
        # Create a 2D kernel that will be applied to both spatial dimensions
        self.kernel = self.add_weight(
            name='kernel2d',
            shape=(self.kernel_size, self.kernel_size, input_shape[-1], self.filters),
            initializer='glorot_uniform',
            trainable=True
        )
        self.a1 = self.add_weight(shape=(self.filters,), initializer='zeros', trainable=True)#, name='h0_bias')
        self.a2 = self.add_weight(shape=(self.filters,), initializer='zeros', trainable=True)#, name='h0_bias')


    def call(self, x):
        print(x.shape)
        input_size = x.shape[1]

        # outer product of x
        x2 = tf.einsum('bijp,bklp->bijklp',x,x)
        print('+',x2.shape)
        
        # first order coeffs.
        # h1_out = self.conv2d(x)
        h1_out = tf.nn.convolution(x, self.kernel, padding='SAME')
        
        # 2nd order coeffs
        # h2_out = self.conv4d(x2)#, axis=-1))
        h2_out = self.__separable_conv4d(x2)
        print('h1 h2 out', h1_out.shape, h2_out.shape)

        # reduced quadratic terms: iterative block summation
        h2_out = [[
            tf.einsum('bijklc->bc', h2_out[:, 0:n1+1, 0:n2+1, 0:n1+1, 0:n2+1, :])
            for n1 in range(input_size)]
            for n2 in range(input_size)
        ]
        h2_out = tf.stack([tf.stack(inner_list, axis=1) for inner_list in h2_out], axis=2)
        print('h2_out shape update', h2_out.shape)
              
        # system output
        y = self.a0 + self.a1*h1_out + self.a2*h2_out # + h3_out #+ h4_out + h5_out
        return y

    
    # Function: 4D separable convolution using a single 2D kernel
    def __separable_conv4d(self, x):
        # x: shape [B, N1, N2, N3, N4, C]
        # B, N1, N2, N3, N4, C = x.shape
        # Get static shape for dimensions that shouldn't change
        input_shape = x.shape.as_list()
        N1, N2, N3, N4 = input_shape[1], input_shape[2], input_shape[3], input_shape[4]
        
        # Get dynamic batch size
        B = tf.shape(x)[0]

        ## Step 1: Convolve over (N1, N2)
        x1 = tf.reshape(x, [-1, N1, N2, self.inchannels])  # shape: (B*N3*N4, N1, N2, C)
        y1 = tf.nn.convolution(x1, self.kernel, padding='SAME')
        y1 = tf.reshape(y1, [B, N1, N2, N3, N4, self.filters])   # (B, N1, N2, N3, N4, C)
        y1 = tf.transpose(y1, perm=[0,3,4,1,2,5])
    
        
        # print('+y1 ', y1.shape)


        ## Step 2: Convolve over (N3, N4)
        x2 = tf.reshape(y1, [-1, N3, N4, self.filters])  # shape: (B*N1*N2, N3, N4, C)
        y2 = tf.nn.convolution(x2, self.kernel, padding='SAME')
        y2 = tf.reshape(y2, [B, N1, N2, N3, N4, self.filters])    # final shape
        y2 = tf.transpose(y2, perm=[0,3,4,1,2,5])

        return y2
        
    def get_kernel(self):
        """Returns the kernel weights as a numpy array"""
        return self.kernel.numpy()

    def get_config(self):
        config = super(NLSI2DVolterra, self).get_config()
        config.update({
            'filters': self.filters,
            'kernel_size': self.kernel_size
        })
        return config

    # def get_config(self):
    #     config = super(NLSI2DVolterra, self).get_config()
    #     return config



if __name__ =='__main__':

    # Define input shape and build the model for summary
    input_shape = (16, 16, 2)  # Replace N with the actual size of x
    inputs = tf.keras.Input(shape=input_shape)

    # Create an instance of the custom layer
    H = NLSI2DVolterra(filters=8)
    #conv1d_filters=32, conv1d_kernel_size=3, 
    #  conv2d_filters=32, conv2d_kernel_size=3)

    # Apply the custom layer to the inputs
    outputs = H(inputs)

    # Build the model
    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Print the model summary
    model.summary()

(None, 16, 16, 2)
+ (None, 16, 16, 16, 16, 2)
h1 h2 out (None, 16, 16, 8) (None, 16, 16, 16, 16, 8)
h2_out shape update (None, 16, 16, 8)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 16, 16, 2)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nlsi2d_volterra                 │ (None, 16, 16, 8)      │           168 │
│ (NLSI2DVolterra)                │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 168 (672.00 B)

 Trainable params: 168 (672.00 B)

 Non-trainable params: 0 (0.00 B)

---

The below code is the earlier implementation of QSI 2D volterra kernel and is incorret.
 
The higher dim convolution filters shape is wrong. Also tf doesnot support more than 3d convlution with tf.nn.convolution and tf.signal.convnd

In [8]:
# #%%
# import tensorflow as tf
# from ConvNDv0 import ConvND

# class NLSI2DVolterra(tf.keras.layers.Layer):
#     """2D input-output NLSI system with 2nd order Volterra approximation --kkt@30-08-2024"""
#     def __init__(
#         self, 
#         filters=1, 
#         kernel_size=3,
#         **kwargs):
#         super(NLSI2DVolterra, self).__init__(**kwargs)
#         self.conv2d_filters = filters
#         self.conv2d_kernel_size = kernel_size
     

#     def build(self, input_shape):
#         # Initialize the bias (a0) for the numerator polynomial
#         self.h0 = self.add_weight(shape=(), initializer='zeros', trainable=True)#, name='h0_bias')
               
#         # Initialize Conv2D layer
#         self.conv2d = tf.keras.layers.SeparableConv2D(
#             filters=self.conv2d_filters, 
#             kernel_size=self.conv2d_kernel_size, 
#             padding='same')
        
#         # Initialize Conv4D layer
#         self.conv4d_filters = self.conv2d_filters
#         self.conv4d_kernel_size = self.conv2d_kernel_size
#         self.conv4d = ConvND(
#             filters=self.conv4d_filters, 
#             kernel_size=self.conv4d_kernel_size)
        

#     def call(self, x):
#         print(x.shape)
#         input_size = x.shape[1]

#         # outer product of x
#         x2 = tf.einsum('bijp,bklp->bijklp',x,x)
#         print('+',x2.shape)
        
#         # first order coeffs.
#         h1_out = self.conv2d(x)
        
#         # 2nd order coeffs
#         h2_out = self.conv4d(x2)#, axis=-1))
#         print('h1 h2 out', h1_out.shape, h2_out.shape)

#         # reduced quadratic terms: iterative block summation
#         h2_out = [[
#             tf.einsum('bijklc->bc', h2_out[:, 0:n1+1, 0:n2+1, 0:n1+1, 0:n2+1, :])
#             for n1 in range(input_size)]
#             for n2 in range(input_size)
#         ]
#         h2_out = tf.stack([tf.stack(inner_list, axis=1) for inner_list in h2_out], axis=2)
#         print('h2_out shape update', h2_out.shape)
              
#         # system output
#         y = self.h0 + h1_out + h2_out# + h3_out #+ h4_out + h5_out
#         return y


#     def get_config(self):
#         config = super(NLSI2DVolterra, self).get_config()
#         return config



# if __name__ =='__main__':

#     # Define input shape and build the model for summary
#     input_shape = (16,16, 2)  # Replace N with the actual size of x
#     inputs = tf.keras.Input(shape=input_shape)

#     # Create an instance of the custom layer
#     H = NLSI2DVolterra(filters=3)
#     #conv1d_filters=32, conv1d_kernel_size=3, 
#     #  conv2d_filters=32, conv2d_kernel_size=3)

#     # Apply the custom layer to the inputs
#     outputs = H(inputs)

#     # Build the model
#     model = tf.keras.Model(inputs=inputs, outputs=outputs)

#     # Print the model summary
#     model.summary()

In [ ]:

# Let's suppose it's called blockwise_diagonal_sum


In [29]:
#%%
import time
import tensorflow as tf
# from ConvNDv0 import ConvND

class NLSI2DVolterra(tf.keras.layers.Layer):
    """2D input-output NLSI system with 2nd order Volterra approximation 
    
    Limitation: Be careful with the # of filters

    --kkt@06-06-2025"""
    def __init__(
        self, 
        filters=1, 
        kernel_size=3,
        **kwargs):
        super(NLSI2DVolterra, self).__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        blocksum_op = tf.load_op_library('./blocksum_op.so')
        self.blockwise_diagonal_sum = blocksum_op.blockwise_diagonal_sum
     

    def build(self, input_shape):
        self.inchannels = input_shape[-1]
        # self.filters = input_shape[-1]

        ## h0:= 0th order term
        # Initialize the bias (a0) for the numerator polynomial
        self.a0 = self.add_weight(shape=(self.filters,), initializer='zeros', trainable=True)#, name='h0_bias')

        ## h:= UIR kernle       
        # Create a 2D kernel that will be applied to both spatial dimensions
        self.kernel = self.add_weight(
            name='kernel2d',
            shape=(self.kernel_size, self.kernel_size, input_shape[-1], self.filters),
            initializer='glorot_uniform',
            trainable=True
        )
        self.a1 = self.add_weight(shape=(self.filters,), initializer='zeros', trainable=True)#, name='h0_bias')
        self.a2 = self.add_weight(shape=(self.filters,), initializer='zeros', trainable=True)#, name='h0_bias')


    def call(self, x):
        print(x.shape)
        input_size = x.shape[1]

        # outer product of x
        x2 = tf.einsum('bijp,bklp->bijklp',x,x)
        print('+',x2.shape)
        
        # first order coeffs.
        # h1_out = self.conv2d(x)
        h1_out = tf.nn.convolution(x, self.kernel, padding='SAME')
        
        # 2nd order coeffs
        # h2_out = self.conv4d(x2)#, axis=-1))
        h2_out = self.__separable_conv4d(x2)
        print('h1 h2 out', h1_out.shape, h2_out.shape)

        #method1
        start = time.time()
        # reduced quadratic terms: iterative block summation
        h2_out_1 = [[
            tf.einsum('bijklc->bc', h2_out[:, 0:n1+1, 0:n2+1, 0:n1+1, 0:n2+1, :])
            for n1 in range(input_size)]
            for n2 in range(input_size)]
        h2_out_1 = tf.stack([tf.stack(inner_list, axis=1) for inner_list in h2_out_1], axis=2)
        print('method1: h2_out shape update', h2_out_1.shape)
        end = time.time()
        elapsed = end - start
        print(f"method 1 Elapsed time: {elapsed:.6f} seconds")
        del elapsed, end, start
        
        
        #method2
        start = time.time()
        h2_out_2 = self.blockwise_diagonal_sum(h2_out)
        print('method2: h2_out shape update', h2_out_2.shape)
        end = time.time()
        elapsed = end - start
        print(f"method 2 Elapsed time: {elapsed:.6f} seconds")
        del elapsed, end, start
              
        # system output
        y1 = self.a0 + self.a1*h1_out + self.a2*h2_out_1 # + h3_out #+ h4_out + h5_out
        y2 = self.a0 + self.a1*h1_out + self.a2*h2_out_2 # + h3_out #+ h4_out + h5_out

        # y = self.a0 + self.a1*h1_out + self.a2*h2_out # + h3_out #+ h4_out + h5_out
        return y1, y2

    
    # Function: 4D separable convolution using a single 2D kernel
    def __separable_conv4d(self, x):
        # x: shape [B, N1, N2, N3, N4, C]
        # B, N1, N2, N3, N4, C = x.shape
        # Get static shape for dimensions that shouldn't change
        input_shape = x.shape.as_list()
        N1, N2, N3, N4 = input_shape[1], input_shape[2], input_shape[3], input_shape[4]
        
        # Get dynamic batch size
        B = tf.shape(x)[0]

        ## Step 1: Convolve over (N1, N2)
        x1 = tf.reshape(x, [-1, N1, N2, self.inchannels])  # shape: (B*N3*N4, N1, N2, C)
        y1 = tf.nn.convolution(x1, self.kernel, padding='SAME')
        y1 = tf.reshape(y1, [B, N1, N2, N3, N4, self.filters])   # (B, N1, N2, N3, N4, C)
        y1 = tf.transpose(y1, perm=[0,3,4,1,2,5])
    
        
        # print('+y1 ', y1.shape)


        ## Step 2: Convolve over (N3, N4)
        x2 = tf.reshape(y1, [-1, N3, N4, self.filters])  # shape: (B*N1*N2, N3, N4, C)
        y2 = tf.nn.convolution(x2, self.kernel, padding='SAME')
        y2 = tf.reshape(y2, [B, N1, N2, N3, N4, self.filters])    # final shape
        y2 = tf.transpose(y2, perm=[0,3,4,1,2,5])

        return y2
        
    def get_kernel(self):
        """Returns the kernel weights as a numpy array"""
        return self.kernel.numpy()

    def get_config(self):
        config = super(NLSI2DVolterra, self).get_config()
        config.update({
            'filters': self.filters,
            'kernel_size': self.kernel_size
        })
        return config

    # def get_config(self):
    #     config = super(NLSI2DVolterra, self).get_config()
    #     return config



if __name__ =='__main__':

    # Define input shape and build the model for summary
    N, channels = 16, 8
    input_shape = (N, N, channels)  # Replace N with the actual size of x
    inputs = tf.keras.Input(shape=input_shape)

    # Create an instance of the custom layer
    H = NLSI2DVolterra(filters=8)
    #conv1d_filters=32, conv1d_kernel_size=3, 
    #  conv2d_filters=32, conv2d_kernel_size=3)

    # Apply the custom layer to the inputs
    outputs = H(inputs)

    # Build the model
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(jit_compile=True)
    # model.compile(jit_compile=False)


    # Print the model summary
    model.summary()

(None, 16, 16, 8)
+ (None, 16, 16, 16, 16, 8)
h1 h2 out (None, 16, 16, 8) (None, 16, 16, 16, 16, 8)
method1: h2_out shape update (None, 16, 16, 8)
method 1 Elapsed time: 0.320719 seconds
method2: h2_out shape update (None, 16, 16, 8)
method 2 Elapsed time: 0.000487 seconds


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_19 (InputLayer)     │ (None, 16, 16, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nlsi2d_volterra_19              │ [(None, 16, 16, 8),    │           600 │
│ (NLSI2DVolterra)                │ (None, 16, 16, 8)]     │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 600 (2.34 KB)

 Trainable params: 600 (2.34 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
import numpy as np
# batch, N, channels = 1, , 1
batch = 4
x = np.arange(batch * N * N * channels).reshape((batch, N, N, channels)).astype(np.float32)
# x = np.arange(batch * N * N * N * N * channels).reshape((batch, N, N, N, N, channels)).astype(np.float32)
x.shape

(4, 16, 16, 8)

In [31]:
# h = NLSI2DVolterra(filters=1)
# h.build(1,N,N,1)
y1, y2 = model.predict(x)
y1.shape, y2.shape

(4, 16, 16, 8)
+ (4, 16, 16, 16, 16, 8)
h1 h2 out (4, 16, 16, 8) (4, 16, 16, 16, 16, 8)
method1: h2_out shape update (4, 16, 16, 8)
method 1 Elapsed time: 0.577274 seconds
method2: h2_out shape update (4, 16, 16, 8)
method 2 Elapsed time: 0.001065 seconds


I0000 00:00:1750592084.872324 1490341 service.cc:148] XLA service 0x7f7d980041e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750592084.872384 1490341 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2025-06-22 17:04:44.992474: W tensorflow/core/framework/op_kernel.cc:1841] OP_REQUIRES failed at xla_ops.cc:577 : INVALID_ARGUMENT: Detected unsupported operations when trying to compile graph __inference_one_step_on_data_23824[] on XLA_GPU_JIT: BlockwiseDiagonalSum (No registered 'BlockwiseDiagonalSum' OpKernel for XLA_GPU_JIT devices compatible with node {{node functional_7_1/nlsi2d_volterra_19_1/BlockwiseDiagonalSum}}){{node functional_7_1/nlsi2d_volterra_19_1/BlockwiseDiagonalSum}}
The op is created at: 
File "<frozen runpy>", line 198, in _run_module_as_main
File "<frozen runpy>", line 88, in _run_code
File "/data1/kishoretarafdar/miniforge3/envs/tf218/lib/python3.12/site-packages/ipykernel_la

InvalidArgumentError: Graph execution error:

Detected at node functional_7_1/nlsi2d_volterra_19_1/BlockwiseDiagonalSum defined at (most recent call last):
<stack traces unavailable>
Detected at node functional_7_1/nlsi2d_volterra_19_1/BlockwiseDiagonalSum defined at (most recent call last):
<stack traces unavailable>
Detected unsupported operations when trying to compile graph __inference_one_step_on_data_23824[] on XLA_GPU_JIT: BlockwiseDiagonalSum (No registered 'BlockwiseDiagonalSum' OpKernel for XLA_GPU_JIT devices compatible with node {{node functional_7_1/nlsi2d_volterra_19_1/BlockwiseDiagonalSum}}){{node functional_7_1/nlsi2d_volterra_19_1/BlockwiseDiagonalSum}}
The op is created at: 
File "<frozen runpy>", line 198, in _run_module_as_main
File "<frozen runpy>", line 88, in _run_code
File "/data1/kishoretarafdar/miniforge3/envs/tf218/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
File "/data1/kishoretarafdar/miniforge3/envs/tf218/lib/python3.12/site-p

In [25]:
np.allclose(y1, y2, rtol=1e-2, atol=1e-2)

True